# TP 1 — Vision artificielle (Master 2 GNIPIA)
**ENS Rabat — Université Mohammed V** · Chapitre 1 : image numérique, couleur, histogrammes, quantification, stockage

Ce notebook réalise les **6 activités** du TP 1 : inspection du tableau image, couleurs et ROI, histogrammes,
quantification, masque de couleur rouge, stockage et fichiers.

> **Images utilisées** : placez vos deux images dans le dossier `data/` sous les noms
> `image1` et `image2` (extension `.jpg`, `.jpeg` ou `.png`).
> - `image1` : image principale (activités 1 à 6)
> - `image2` : autre image pour tester le masque rouge (activité 5, question 4)
>
> Si elles sont absentes, des scènes synthétiques sont générées automatiquement.
> Les valeurs numériques sont **calculées par le code** à chaque exécution ; les commentaires
> de réponse donnent le raisonnement (valable pour toute image).

In [ ]:
from pathlib import Path
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 100
dossier = Path("data")
sorties = Path("sorties")
LARGEUR_MAX = 640   # largeur maximale (px) ; mettre None pour garder la taille d'origine
dossier.mkdir(exist_ok=True)
sorties.mkdir(exist_ok=True)

print("OpenCV", cv.__version__, "| NumPy", np.__version__)

### Préparation — images d'exemple (générées si absentes)

In [ ]:
from pathlib import Path

dossier = Path("data")
dossier.mkdir(exist_ok=True)

def scene_synthetique(luminosite=1.0, objet="voiture", graine=0):
    """Scène 480x640 BGR : ciel, route, voiture (ou balle) rouge, panneau rouge, arbre vert."""
    rng = np.random.default_rng(graine)
    h, w = 480, 640
    img = np.zeros((h, w, 3), np.uint8)
    # ciel en dégradé
    for y in range(240):
        t = y / 240
        img[y, :] = (int(235 - 40 * t), int(200 - 30 * t), int(120 + 20 * t))   # BGR
    img[240:] = (90, 90, 90)                                  # route grise
    cv.rectangle(img, (0, 240), (w, 262), (60, 140, 60), -1)  # bas-côté vert
    for x in range(20, w, 120):                               # lignes blanches
        cv.rectangle(img, (x, 380), (x + 60, 390), (235, 235, 235), -1)
    cv.rectangle(img, (80, 150), (100, 250), (30, 70, 90), -1)         # tronc
    cv.circle(img, (90, 130), 45, (40, 150, 50), -1)                   # feuillage
    cv.rectangle(img, (520, 130), (528, 250), (150, 150, 150), -1)     # poteau
    cv.circle(img, (524, 110), 32, (30, 30, 200), -1)                  # panneau rouge
    cv.circle(img, (524, 110), 20, (240, 240, 240), -1)
    if objet == "voiture":
        cv.rectangle(img, (230, 300), (450, 380), (25, 25, 200), -1)    # carrosserie rouge
        cv.rectangle(img, (275, 262), (410, 305), (25, 25, 200), -1)    # habitacle
        cv.rectangle(img, (288, 272), (340, 300), (220, 200, 160), -1)  # vitres
        cv.rectangle(img, (350, 272), (398, 300), (220, 200, 160), -1)
        cv.circle(img, (280, 385), 24, (20, 20, 20), -1)                # roues
        cv.circle(img, (400, 385), 24, (20, 20, 20), -1)
    else:
        cv.circle(img, (320, 340), 55, (40, 40, 210), -1)               # balle rouge
        cv.rectangle(img, (500, 290), (620, 400), (30, 100, 220), -1)   # mur orangé
    img = img.astype(np.float32) * luminosite
    img += rng.normal(0, 3, img.shape)                                  # bruit capteur
    return np.clip(img, 0, 255).astype(np.uint8)

def trouver_image(nom):
    for ext in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"):
        p = dossier / (nom + ext)
        if p.exists():
            return p
    return None

def charger(chemin):
    """Charge en BGR et réduit si la largeur dépasse LARGEUR_MAX (INTER_AREA)."""
    img = cv.imread(str(chemin))
    if img is None:
        raise FileNotFoundError(f"Image introuvable ou illisible : {chemin}")
    if LARGEUR_MAX and img.shape[1] > LARGEUR_MAX:
        f = LARGEUR_MAX / img.shape[1]
        img = cv.resize(img, (LARGEUR_MAX, round(img.shape[0] * f)), interpolation=cv.INTER_AREA)
    return img

if trouver_image("image1") is None:
    cv.imwrite(str(dossier / "image1.jpg"), scene_synthetique(), [cv.IMWRITE_JPEG_QUALITY, 95])
    print("image1 absente -> scène synthétique générée : data/image1.jpg")
if trouver_image("image2") is None:
    cv.imwrite(str(dossier / "image2.jpg"), scene_synthetique(0.6, "balle", 1), [cv.IMWRITE_JPEG_QUALITY, 95])
    print("image2 absente -> scène synthétique générée : data/image2.jpg")

chemin1, chemin2 = trouver_image("image1"), trouver_image("image2")
print("image1 :", chemin1, "| image2 :", chemin2)

---
## Activité 1 — Inspection du tableau image

In [ ]:
bgr = cv.imread(str(chemin1))
if bgr is None:
    raise FileNotFoundError("Image introuvable ou illisible")
if LARGEUR_MAX and bgr.shape[1] > LARGEUR_MAX:          # réduction éventuelle (photos haute résolution)
    bgr = charger(chemin1)
h, w, c = bgr.shape
print(bgr.shape, bgr.dtype, bgr.size, bgr.nbytes)

In [ ]:
# 1. H×W et H×W×C comparés à bgr.size
print("H × W         =", h * w, "pixels")
print("H × W × C     =", h * w * c, "valeurs")
print("bgr.size      =", bgr.size, "->", bgr.size == h * w * c)

# 2. Pixel central et ordre des composantes
pixel_central = bgr[h // 2, w // 2]
print("Pixel central (ligne, colonne) =", (h // 2, w // 2), "->", pixel_central, "ordre : B, G, R")

# 3. Relation forme / type / nbytes
attendu = h * w * c * bgr.dtype.itemsize
print(f"{h} × {w} × {c} × {bgr.dtype.itemsize} octet = {attendu} octets | bgr.nbytes = {bgr.nbytes} -> {attendu == bgr.nbytes}")

**Réponses**
1. `H × W` compte les **pixels** ; `H × W × C` compte les **valeurs** (un pixel couleur = C = 3 valeurs). `bgr.size` renvoie le nombre de valeurs : `bgr.size = H × W × C`.
2. `bgr[h//2, w//2]` renvoie un vecteur de 3 valeurs dans l'ordre **B, G, R** : OpenCV charge les images en **BGR** et non en RGB. L'indexation se fait en `[ligne, colonne]` = `[y, x]`.
3. `bgr.nbytes = H × W × C × itemsize`, avec `itemsize = 1` octet pour `uint8`. Pour une image 480 × 640 × 3 : 921 600 octets.

---
## Activité 2 — Couleurs et région d'intérêt

In [ ]:
rgb = cv.cvtColor(bgr, cv.COLOR_BGR2RGB)
gris = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
hsv = cv.cvtColor(bgr, cv.COLOR_BGR2HSV)
roi = rgb[h // 4: 3 * h // 4, w // 4: 3 * w // 4].copy()

# Trois canaux RGB affichés séparément en niveaux de gris
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
axes[0].imshow(rgb); axes[0].set_title("RGB")
for i, nom in enumerate(["R", "G", "B"]):
    axes[i + 1].imshow(rgb[..., i], cmap="gray", vmin=0, vmax=255)
    axes[i + 1].set_title(f"Canal {nom}")
for a in axes: a.axis("off")
plt.tight_layout()
plt.savefig(sorties / "canaux_rgb.png", dpi=120)
plt.show()

In [ ]:
# 2. Forme de la ROI + justification de copy()
print("Forme ROI :", roi.shape)
vue = rgb[h // 4: 3 * h // 4, w // 4: 3 * w // 4]
print("Sans copy(), partage la mémoire avec rgb :", np.shares_memory(vue, rgb))
print("Avec copy(), partage la mémoire          :", np.shares_memory(roi, rgb))

# 3. Réduction à 1/4 de chaque dimension avec INTER_AREA
petite = cv.resize(rgb, (w // 4, h // 4), interpolation=cv.INTER_AREA)
print("Nouvelle forme :", petite.shape)
print("Rapport des nombres de pixels :", (h * w) / (petite.shape[0] * petite.shape[1]))

In [ ]:
# 4. Comparaison légendée : image, gris, ROI
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(rgb);                                  axes[0].set_title(f"Image RGB {rgb.shape}")
axes[1].imshow(gris, cmap="gray", vmin=0, vmax=255);  axes[1].set_title(f"Niveaux de gris {gris.shape}")
axes[2].imshow(roi);                                  axes[2].set_title(f"ROI centrale {roi.shape}")
for a in axes: a.axis("off")
plt.tight_layout()
plt.savefig(sorties / "comparaison_image_gris_roi.png", dpi=120)
plt.show()

**Réponses**
1. OpenCV range les canaux en **BGR**, Matplotlib attend du **RGB**. Sans conversion, rouge et bleu sont inversés à l'affichage. La conversion ne sert donc qu'à l'**affichage** ; le traitement peut rester en BGR.
2. ROI = moitié de la hauteur et moitié de la largeur : pour 480 × 640 → **(240, 320, 3)**. Un découpage NumPy est une **vue** (même mémoire que l'image d'origine) : modifier la ROI modifierait `rgb`. `copy()` crée un tableau **indépendant**, ce qui protège l'image source.
3. Réduction à 1/4 sur chaque axe : (480, 640, 3) → **(120, 160, 3)**. Le rapport des nombres de pixels est 4 × 4 = **16**. `INTER_AREA` moyenne les pixels couverts : c'est l'interpolation recommandée pour **réduire** (limite l'aliasing).
4. Figures enregistrées dans `sorties/`.

---
## Activité 3 — Intensités et histogrammes

In [ ]:
clair = np.clip(gris.astype(np.float32) + 40, 0, 255)
clair = clair.astype(np.uint8)
contraste = np.clip(
    0.4 * (gris.astype(np.float32) - 128) + 128, 0, 255
).astype(np.uint8)

# Pourquoi convertir en float ? Démonstration du débordement uint8
a = np.array([230], np.uint8)
print("uint8 : 230 + 40 =", (a + 40)[0], "(débordement modulo 256)")
print("float : 230 + 40 =", float(a.astype(np.float32)[0] + 40), "-> clip à 255")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8), gridspec_kw={"width_ratios": [1, 1, 1, 1.6]})
for a, im, t in zip(axes[:3], [gris, clair, contraste], ["Original (gris)", "Éclairci (+40)", "Contraste réduit (×0,4)"]):
    a.imshow(im, cmap="gray", vmin=0, vmax=255); a.set_title(t); a.axis("off")

for im, t in zip([gris, clair, contraste], ["Original", "Éclairci +40", "Contraste ×0,4"]):
    axes[3].hist(im.ravel(), bins=32, range=(0, 256), histtype="step", linewidth=2, label=t)
axes[3].set_xlabel("Intensité"); axes[3].set_ylabel("Effectif"); axes[3].set_title("Histogrammes (32 classes)")
axes[3].legend()
plt.tight_layout()
plt.savefig(sorties / "histogrammes.png", dpi=120)
plt.show()

In [ ]:
for nom, im in [("original", gris), ("éclairci", clair), ("contraste", contraste)]:
    print(f"{nom:10s} moyenne={im.mean():6.1f}  écart-type={im.std():6.1f}  "
          f"min={im.min():3d}  max={im.max():3d}  "
          f"%pixels=255: {100*np.mean(im==255):5.2f}  %pixels=0: {100*np.mean(im==0):5.2f}")

**Réponses**
1. Histogrammes tracés sur les mêmes axes (32 classes de largeur 8 sur [0, 256[).
2. **Éclaircissement (+40)** : tout l'histogramme est **décalé vers la droite** (moyenne + 40) sans changer sa forme. Un **écrêtage** apparaît à droite : toutes les valeurs > 215 deviennent 255 et s'accumulent dans la dernière classe (information perdue, irréversible). **Contraste réduit** : l'histogramme est **resserré** autour de 128 (écart-type multiplié par 0,4) ; les écarts entre intensités diminuent. Ici, l'écrêtage est négligeable car 0,4·(v−128)+128 reste dans [77, 179].
3. Ajouter une constante ne crée **aucune information** : les détails déjà saturés (zones à 255) ou noyés dans le bruit ne sont pas récupérés ; on perd même de la dynamique dans les hautes lumières, et le bruit est éclairci avec le signal. Une image sous-exposée acquise avec peu de lumière reste donc peu exploitable : mieux vaut corriger **à l'acquisition** (exposition, éclairage).

---
## Activité 4 — Quantification sur quatre niveaux

In [ ]:
L = 4
q = np.rint(gris.astype(np.float32) * (L - 1) / 255)
q = np.rint(q * 255 / (L - 1)).astype(np.uint8)
print(np.unique(q), q.nbytes)
print("gris.nbytes =", gris.nbytes, "| q.nbytes =", q.nbytes, "| identiques :", q.nbytes == gris.nbytes)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(gris, cmap="gray", vmin=0, vmax=255); axes[0].set_title("Original (256 niveaux)")
axes[1].imshow(q, cmap="gray", vmin=0, vmax=255);    axes[1].set_title("Quantifiée (4 niveaux)")
axes[2].hist([gris.ravel(), q.ravel()], bins=32, range=(0, 256), label=["Original", "Quantifiée"])
axes[2].set_title("Histogrammes"); axes[2].legend()
for a in axes[:2]: a.axis("off")
plt.tight_layout()
plt.savefig(sorties / "quantification_4_niveaux.png", dpi=120)
plt.show()

In [ ]:
# Stockage réellement compact : 2 bits par pixel (4 pixels par octet)
idx = np.rint(gris.astype(np.float32) * (L - 1) / 255).astype(np.uint8).ravel()   # indices 0..3
manque = (-idx.size) % 4
idx_p = np.pad(idx, (0, manque))
compact = (idx_p[0::4] << 6) | (idx_p[1::4] << 4) | (idx_p[2::4] << 2) | idx_p[3::4]
print("Tableau uint8 :", gris.nbytes, "octets | tableau compact 2 bits :", compact.nbytes, "octets",
      f"(facteur {gris.nbytes / compact.nbytes:.2f})")

# Décodage et vérification
dec = np.empty(idx_p.size, np.uint8)
dec[0::4] = (compact >> 6) & 3; dec[1::4] = (compact >> 4) & 3
dec[2::4] = (compact >> 2) & 3; dec[3::4] = compact & 3
rec = (dec[:idx.size].reshape(gris.shape) * (255 // (L - 1))).astype(np.uint8)   # 0,85,170,255 (255//3 = 85)
print("Reconstruction identique à q :", np.array_equal(rec, q))

**Réponses**
1. L'image quantifiée présente des **paliers** (faux contours / *banding*) : les dégradés lisses (ciel, route) deviennent des bandes uniformes séparées par des contours abrupts, et les détails fins à faible contraste disparaissent.
2. 4 niveaux = 2² : `L = 2^b`. Il suffit théoriquement de **b = log₂(4) = 2 bits par pixel**.
3. `q` reste un tableau `uint8` : NumPy alloue **1 octet (8 bits) par élément**, quelle que soit la plage réelle des valeurs. Seul le **nombre de valeurs distinctes** a diminué, pas la taille du type : `q.nbytes == gris.nbytes`.
4. Il faut **empaqueter les bits** : stocker les indices 0–3 sur 2 bits et placer 4 pixels par octet (décalages `<<` / `np.packbits` / format binaire dédié). On obtient alors une taille divisée par 4 (cellule ci-dessus), au prix d'un décodage avant tout calcul.

---
## Activité 5 — Masque de couleur rouge
> Dans OpenCV (`uint8`) : **H ∈ [0, 179]**, S ∈ [0, 255], V ∈ [0, 255]. Le rouge est proche de H = 0 **et** H = 179.

In [ ]:
m1 = cv.inRange(hsv, np.array([0, 80, 50], np.uint8),
                     np.array([10, 255, 255], np.uint8))
m2 = cv.inRange(hsv, np.array([170, 80, 50], np.uint8),
                     np.array([179, 255, 255], np.uint8))
masque = cv.bitwise_or(m1, m2)
selection = cv.bitwise_and(rgb, rgb, mask=masque)
taux = 100 * np.count_nonzero(masque) / masque.size
print(f"Pixels sélectionnés : {np.count_nonzero(masque)} / {masque.size} = {taux:.2f} %")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(rgb);                              axes[0].set_title("Image")
axes[1].imshow(masque, cmap="gray");              axes[1].set_title(f"Masque rouge ({taux:.1f} %)")
axes[2].imshow(selection);                        axes[2].set_title("Sélection")
for a in axes: a.axis("off")
plt.tight_layout()
plt.savefig(sorties / "masque_rouge.png", dpi=120)
plt.show()

In [ ]:
# 3. Variation d'UN seuil à la fois (les autres restent aux valeurs de référence)
def taux_rouge(h_max=10, h_min=170, s_min=80, v_min=50):
    a = cv.inRange(hsv, np.array([0, s_min, v_min], np.uint8), np.array([h_max, 255, 255], np.uint8))
    b = cv.inRange(hsv, np.array([h_min, s_min, v_min], np.uint8), np.array([179, 255, 255], np.uint8))
    return 100 * np.count_nonzero(cv.bitwise_or(a, b)) / a.size

print(f"Référence (H≤10, H≥170, S≥80, V≥50)  : {taux_rouge():.2f} %")
for s in (40, 120, 160):
    print(f"S_min = {s:3d} (autres inchangés)       : {taux_rouge(s_min=s):.2f} %")
for v in (20, 100, 150):
    print(f"V_min = {v:3d} (autres inchangés)       : {taux_rouge(v_min=v):.2f} %")
for hm in (5, 15, 25):
    print(f"H_max = {hm:3d} (autres inchangés)       : {taux_rouge(h_max=hm):.2f} %")

In [ ]:
# 4. Autre image autorisée (plus sombre, objet rouge différent)
bgr2 = charger(chemin2)
rgb2 = cv.cvtColor(bgr2, cv.COLOR_BGR2RGB)
hsv2 = cv.cvtColor(bgr2, cv.COLOR_BGR2HSV)
mk2 = cv.bitwise_or(
    cv.inRange(hsv2, np.array([0, 80, 50], np.uint8),   np.array([10, 255, 255], np.uint8)),
    cv.inRange(hsv2, np.array([170, 80, 50], np.uint8), np.array([179, 255, 255], np.uint8)))
t2 = 100 * np.count_nonzero(mk2) / mk2.size
print(f"image2 : {t2:.2f} % de pixels sélectionnés")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(rgb2); axes[0].set_title("image2")
axes[1].imshow(mk2, cmap="gray"); axes[1].set_title(f"Masque ({t2:.1f} %)")
axes[2].imshow(cv.bitwise_and(rgb2, rgb2, mask=mk2)); axes[2].set_title("Sélection")
for a in axes: a.axis("off")
plt.tight_layout()
plt.savefig(sorties / "masque_rouge_image2.png", dpi=120)
plt.show()

**Réponses**
1. Image, masque et sélection affichés ; le pourcentage de pixels sélectionnés est `taux` (cellule ci-dessus).
2. **Zone pertinente** : l'objet rouge visé (par ex. la carrosserie d'une voiture rouge) est sélectionné. **Zone sans rapport avec un objet cible précis** : tout autre élément rouge (panneau, vêtement, enseigne, brique rouge…), les reflets ou ombres rougeâtres, ou des teintes orangées si H_max est trop élevé. *(À adapter à ce que votre masque sélectionne réellement.)* Le masque sélectionne une **couleur**, pas un objet.
3. Effets observés (un seul seuil modifié à la fois) :
   - **S_min ↑** : exclut les couleurs pâles/délavées (moins de faux positifs, mais perte de zones peu saturées).
   - **V_min ↑** : exclut les zones sombres (ombres), mais aussi les parties rouges mal éclairées.
   - **H_max ↑** : le masque s'élargit vers l'orange (plus de faux positifs).
   Paramètres retenus : **H ∈ [0, 10] ∪ [170, 179], S ≥ 80, V ≥ 50**.
4. Non : le masque détecte une **plage de teintes**, pas une **catégorie** d'objet. Tout objet rouge (balle, panneau, mur) est sélectionné au même titre qu'une voiture ; et une voiture bleue serait ignorée. L'éclairage modifie en outre S et V (image plus sombre → moins de pixels passent le seuil V).

---
## Activité 6 — Stockage et fichiers

In [ ]:
cv.imwrite(str(sorties / "copie.png"), bgr)
cv.imwrite(str(sorties / "qualite90.jpg"), bgr, [cv.IMWRITE_JPEG_QUALITY, 90])
cv.imwrite(str(sorties / "qualite30.jpg"), bgr, [cv.IMWRITE_JPEG_QUALITY, 30])

print(f"{'Fichier':14s} {'Taille fichier (octets)':>24s} {'Tableau décodé (octets)':>25s} {'Rapport':>9s} {'Pixels identiques':>19s}")
for nom in ["copie.png", "qualite90.jpg", "qualite30.jpg"]:
    chemin = sorties / nom
    relu = cv.imread(str(chemin))
    taille_f = chemin.stat().st_size
    identique = np.array_equal(relu, bgr)
    print(f"{nom:14s} {taille_f:24d} {relu.nbytes:25d} {relu.nbytes / taille_f:9.1f} {str(identique):>19s}")

print("\nTableau BGR initial :", bgr.nbytes, "octets")
for nom in ["qualite90.jpg", "qualite30.jpg"]:
    relu = cv.imread(str(sorties / nom))
    err = np.abs(relu.astype(np.int16) - bgr.astype(np.int16))
    print(f"{nom} : erreur absolue moyenne = {err.mean():.2f}, maximale = {err.max()}")

**Réponses**
1. La **taille du fichier** (compressé) est très inférieure à celle du **tableau décodé**, qui vaut toujours `H × W × C` octets quel que soit le format : une fois relue, l'image est un tableau dense `uint8`. Le rapport dépend du contenu et du format (JPEG qualité 30 < JPEG qualité 90 < PNG en général).
2. `np.array_equal(relu, bgr)` : **True pour le PNG** (compression **sans perte**) ; **False pour les JPEG** (compression **avec perte**, d'autant plus forte que la qualité est basse : l'erreur moyenne augmente quand la qualité diminue).
3. Le PNG compresse sans perte : il restitue **exactement** le tableau qu'on lui donne. Mais si ce tableau provient déjà d'un JPEG, les informations détruites par la quantification DCT et le sous-échantillonnage des chrominances sont **déjà perdues** : le PNG les préserve telles quelles (avec leurs artefacts de blocs) sans pouvoir les restaurer. Enregistrer un JPEG en PNG ne l'« améliore » donc pas ; ça produit seulement un fichier plus gros.

---
## Conclusion
- Une image est un **tableau** `(H, W, C)` : taille mémoire = `H × W × C × itemsize`, indépendante du format de fichier.
- L'**échantillonnage** et la **quantification** détruisent de l'information de façon irréversible (aliasing, paliers).
- L'**histogramme** décrit la distribution des intensités mais **pas** leur organisation spatiale.
- Un **masque HSV** sélectionne une couleur, pas un objet : il est sensible à l'éclairage et ne reconnaît pas de catégorie.
- **PNG** = sans perte ; **JPEG** = avec perte : la compression ne restaure jamais ce qui a déjà été perdu.